In [82]:
#M21-L9

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
#import plotly.express as px

#init data
diabetes_data = pd.read_csv('data/diabetes_data.csv')

display(diabetes_data.info())
diabetes_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               778 non-null    int64  
 1   Glucose                   778 non-null    int64  
 2   BloodPressure             778 non-null    int64  
 3   SkinThickness             778 non-null    int64  
 4   Insulin                   778 non-null    int64  
 5   BMI                       778 non-null    float64
 6   DiabetesPedigreeFunction  778 non-null    float64
 7   Age                       778 non-null    int64  
 8   Outcome                   778 non-null    int64  
 9   Gender                    778 non-null    str    
dtypes: float64(2), int64(7), str(1)
memory usage: 60.9 KB


None

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Gender
0,6,98,58,33,190,34.0,0.430,43,0,Female
1,2,112,75,32,0,35.7,0.148,21,0,Female
2,2,108,64,0,0,30.8,0.158,21,0,Female
3,8,107,80,0,0,24.6,0.856,34,0,Female
4,7,136,90,0,0,29.9,0.210,50,0,Female


Задание 8.1

Начнём с поиска дубликатов в данных. Найдите все повторяющиеся строки в данных и удалите их. Для поиска используйте все признаки в данных. Сколько записей осталось в данных?

In [83]:
#Задание 8.1
display(diabetes_data.shape)
#diabetes_data_duplicates = diabetes_data.duplicated()
diabetes_data_dedupped = diabetes_data.drop_duplicates()
display(diabetes_data_dedupped.shape)



(778, 10)

(768, 10)

Задание 8.2

Далее найдите все неинформативные признаки в данных и избавьтесь от них. В качестве порога информативности возьмите 0.95: удалите все признаки, для которых 95 % значений повторяются или 95 % записей уникальны. В ответ запишите имена признаков, которые вы нашли (без кавычек).

In [84]:
#список неинформативных признаков
low_information_cols = [] 

#цикл по всем столбцам
for col in diabetes_data_dedupped.columns:
    #наибольшая относительная частота в признаке
    top_freq = diabetes_data_dedupped[col].value_counts(normalize=True).max()
    #доля уникальных значений от размера признака
    nunique_ratio = diabetes_data_dedupped[col].nunique() / diabetes_data_dedupped[col].count()
    # сравниваем наибольшую частоту с порогом
    if top_freq > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(top_freq*100, 2)}% одинаковых значений')
    # сравниваем долю уникальных значений с порогом
    if nunique_ratio > 0.95:
        low_information_cols.append(col)
        print(f'{col}: {round(nunique_ratio*100, 2)}% уникальных значений')

diabetes_data_dedup_inform = diabetes_data_dedupped.drop(low_information_cols, axis=1)
print(f'Результирующее число признаков: {diabetes_data_dedup_inform.shape[1]}')

Gender: 100.0% одинаковых значений
Результирующее число признаков: 9


Задание 8.3

Попробуйте найти пропуски в данных с помощью метода isnull().

Спойлер: ничего не найдёте. А они есть! Просто они скрыты от наших глаз. В таблице пропуски в столбцах Glucose, BloodPressure, SkinThickness, Insulin и BMI обозначены нулём, поэтому традиционные методы поиска пропусков ничего вам не покажут. Давайте это исправим!

Замените все записи, равные 0, в столбцах Glucose, BloodPressure, SkinThickness, Insulin и BMI на символ пропуска. Его вы можете взять из библиотеки numpy: np.nan.

Какая доля пропусков содержится в столбце Insulin? Ответ округлите до сотых.

In [85]:
#display(diabetes_data_dedup_inform.isnull())
#display(diabetes_data_dedup_inform.shape)

rowlist = [
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
]

for row_name in rowlist:
    diabetes_data_dedup_inform[row_name] = diabetes_data_dedup_inform[row_name].apply(lambda x: np.nan if x == 0 else x)

display(diabetes_data_dedup_inform['Insulin'].isnull().value_counts(normalize=True).round(2))

Insulin
False    0.51
True     0.49
Name: proportion, dtype: float64

Задание 8.4

Удалите из данных признаки, где число пропусков составляет более 30 %. Сколько признаков осталось в ваших данных (с учетом удаленных неинформативных признаков в задании 8.2)?

In [86]:
#Задание 8.4

display(diabetes_data_dedup_inform.shape)

#создаем копию исходной таблицы
diabetes_data_dedup_inform_drop = diabetes_data_dedup_inform.copy()
#задаем минимальный порог: вычисляем 70% от числа строк
thresh = diabetes_data_dedup_inform_drop.shape[0]*0.7
display(thresh)
#удаляем столбцы, в которых более 30% (100-70) пропусков
diabetes_data_dedup_inform_drop = diabetes_data_dedup_inform_drop.dropna(thresh=thresh, axis=1)

#удаляем записи, в которых есть хотя бы 1 пропуск
#diabetes_data_dedup_inform_drop = diabetes_data_dedup_inform_drop.dropna(how='any', axis=0)

#отображаем результирующую долю пропусков
#diabetes_data_dedup_inform_drop.isnull().mean()
display(diabetes_data_dedup_inform_drop.shape)



(768, 9)

537.5999999999999

(768, 8)

#Задание 8.5
Удалите из данных только те строки, в которых содержится более двух пропусков одновременно. Чему равно результирующее число записей в таблице?

In [87]:
#Задание 8.5

display(diabetes_data_dedup_inform_drop.shape)
#удаляем записи, в которых есть хотя бы 2 пропуск

#отбрасываем строки с числом пропусков более 2 в строке
#m число признаков после удаления столбцов
m = diabetes_data_dedup_inform_drop.shape[1]
diabetes_data_dedup_inform_drop = diabetes_data_dedup_inform_drop.dropna(thresh=m-2,axis=0)

display(diabetes_data_dedup_inform_drop.shape)

(768, 8)

(761, 8)

Задание 8.6
В оставшихся записях замените пропуски на медиану. Чему равно среднее значение в столбце SkinThickness? Ответ округлите до десятых.

In [88]:
#создаем копию исходной таблицы
diabetes_data_dedup_inform_drop_fillna = diabetes_data_dedup_inform_drop.copy()

diabetes_data_dedup_inform_drop_fillna = diabetes_data_dedup_inform_drop_fillna.fillna(diabetes_data_dedup_inform_drop_fillna.median())
display(diabetes_data_dedup_inform_drop_fillna['SkinThickness'].mean())

'''
#Пример
#создаем словарь имя столбца: число(признак) на который надо заменить пропуски
values = {
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
}
#создаем копию исходной таблицы
fill_data = sber_data.copy()
#создаем словарь имя столбца: число(признак) на который надо заменить пропуски
values = {
    'life_sq': fill_data['full_sq'],
    'metro_min_walk': fill_data['metro_min_walk'].median(),
    'metro_km_walk': fill_data['metro_km_walk'].median(),
    'railroad_station_walk_km': fill_data['railroad_station_walk_km'].median(),
    'railroad_station_walk_min': fill_data['railroad_station_walk_min'].median(),
    'hospital_beds_raion': fill_data['hospital_beds_raion'].mode()[0],
    'preschool_quota': fill_data['preschool_quota'].mode()[0],
    'school_quota': fill_data['school_quota'].mode()[0],
    'floor': fill_data['floor'].mode()[0]
}
#заполняем пропуски в соответствии с заявленным словарем
fill_data = fill_data.fillna(values)
#выводим результирующую долю пропусков
fill_data.isnull().mean()
'''


np.float64(29.109067017082786)

"\n#Пример\n#создаем словарь имя столбца: число(признак) на который надо заменить пропуски\nvalues = {\n    'Glucose',\n    'BloodPressure',\n    'SkinThickness',\n    'Insulin',\n    'BMI'\n}\n#создаем копию исходной таблицы\nfill_data = sber_data.copy()\n#создаем словарь имя столбца: число(признак) на который надо заменить пропуски\nvalues = {\n    'life_sq': fill_data['full_sq'],\n    'metro_min_walk': fill_data['metro_min_walk'].median(),\n    'metro_km_walk': fill_data['metro_km_walk'].median(),\n    'railroad_station_walk_km': fill_data['railroad_station_walk_km'].median(),\n    'railroad_station_walk_min': fill_data['railroad_station_walk_min'].median(),\n    'hospital_beds_raion': fill_data['hospital_beds_raion'].mode()[0],\n    'preschool_quota': fill_data['preschool_quota'].mode()[0],\n    'school_quota': fill_data['school_quota'].mode()[0],\n    'floor': fill_data['floor'].mode()[0]\n}\n#заполняем пропуски в соответствии с заявленным словарем\nfill_data = fill_data.fillna(va

Задание 8.7
Сколько выбросов найдёт классический метод межквартильного размаха в признаке SkinThickness?

In [89]:
#Задание 8.7

def outliers_iqr(data, feature):
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 1.5)
    upper_bound = quartile_3 + (iqr * 1.5)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_iqr(diabetes_data_dedup_inform_drop_fillna, 'SkinThickness')
print(f'Число выбросов по методу Тьюки: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу Тьюки: 87
Результирующее число записей: 674


Задание 8.8
Сколько выбросов найдёт классический метод z-отклонения в признаке SkinThickness?

In [90]:
#Задание 8.8

def outliers_z_score(data, feature, log_scale=False):
    if log_scale:
        x = np.log(data[feature]+1)
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - 3 * sigma
    upper_bound = mu + 3 * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers, cleaned = outliers_z_score(diabetes_data_dedup_inform_drop_fillna, 'SkinThickness', log_scale=False)
print(f'Число выбросов по методу z-отклонения: {outliers.shape[0]}')
print(f'Результирующее число записей: {cleaned.shape[0]}')

Число выбросов по методу z-отклонения: 4
Результирующее число записей: 757


Задание 8.9

На приведённой гистограмме показано распределение признака DiabetesPedigreeFunction. Такой вид распределения очень похож на логнормальный, и он заставляет задуматься о логарифмировании признака. Найдите сначала число выбросов в признаке DiabetesPedigreeFunction с помощью классического метода межквартильного размаха.

In [91]:
#Задание 8.9

def outliers_iqr(data, feature):
    x = data[feature]
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 1.5)
    upper_bound = quartile_3 + (iqr * 1.5)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

outliers1, cleaned1 = outliers_iqr(diabetes_data_dedup_inform_drop_fillna, 'DiabetesPedigreeFunction')
print(f'Число выбросов по методу Тьюки: {outliers1.shape[0]}')
print(f'Результирующее число записей: {cleaned1.shape[0]}')


def outliers_iqr_mod(data, feature, left=1.5 , right=1.5, log_scale=False):    
    if log_scale:
        #x = np.log(data[feature]+1)
        x = np.log(data[feature])
    else:
        x = data[feature]
    
    quartile_1, quartile_3 = x.quantile(0.25), x.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * left)
    upper_bound = quartile_3 + (iqr * right)
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned    

outliers2, cleaned2 = outliers_iqr_mod(diabetes_data_dedup_inform_drop_fillna, 'DiabetesPedigreeFunction', log_scale=True)

print(f'Число выбросов по методу z-отклонения: {outliers2.shape[0]}')
print(f'Результирующее число записей: {cleaned2.shape[0]}')

display((outliers1.shape[0] - outliers2.shape[0]))

Число выбросов по методу Тьюки: 29
Результирующее число записей: 732
Число выбросов по методу z-отклонения: 0
Результирующее число записей: 761


29

# Дополнительное задание:

Имеются две базы данных (два листа Excel-файла): база с ценами конкурентов (Data_Parsing) и внутренняя база компании (Data_Company).

Data_TSUM.xlsx

В базе парсинга есть два id, однозначно определяющие товар: producer_id и producer_color.

В базе компании есть два аналогичных поля: item_id и color_id.

Нам известно, что коды в двух базах отличаются наличием набора служебных символов. В базе парсинга встречаются следующие символы: _, -, ~, \\, /.

Необходимо:

Считать данные из Excel в DataFrame (Data_Parsing) и (Data_Company).
Подтянуть к базе парсинга данные из базы компании (item_id, color_id, current_price) и сформировать столбец разницы цен в % (цена конкурента к нашей цене).
Определить сильные отклонения от среднего в разности цен в пределах бренда-категории (то есть убрать случайные выбросы, сильно искажающие сравнение). Критерий — по вкусу, написать комментарий в коде.
Записать новый файл Excel с базой парсинга, приклееными к ней столбцами из пункта 2 и с учётом пункта 3 (можно добавить столбец outlier и проставить Yes для выбросов).